# Standalone Colab Notebook

This notebook is intentionally a single runnable code cell.

How to use:
- open in Google Colab
- edit the config section at the top of the code cell
- run that one code cell

It can:
- optionally mount Google Drive
- optionally pull `krooz0/deep-fake-detection-on-images-and-videos`
- load a dataset from Kaggle, a zip file, or an existing folder
- train a binary `fake` vs `real` image model
- save models, metrics, history, class names, and a run summary


In [ ]:
# =========================
# Config
# =========================
from pathlib import Path

MOUNT_DRIVE = False
PULL_REFERENCE_KERNEL = True

KAGGLE_JSON = Path('/content/kaggle.json')
KAGGLE_KERNEL = 'krooz0/deep-fake-detection-on-images-and-videos'
KAGGLE_DATASET = None  # e.g. 'owner/dataset-slug'; if None, tries kernel metadata source

DATA_DIR = Path('/content/data')
DATASET_ARCHIVE = None  # e.g. Path('/content/your_dataset.zip')
DATASET_SOURCE_DIR = None  # e.g. Path('/content/drive/MyDrive/your_dataset_folder')

WORK_DIR = Path('/content/work')
OUTPUT_DIR = Path('/content/output')

BACKBONE = 'xception'  # xception, vgg16, mobilenetv2, efficientnetb0, custom_cnn
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 8
FINETUNE_EPOCHS = 4
LEARNING_RATE = 1e-4
SEED = 42

# =========================
# Runtime setup
# =========================
import json
import os
import shutil
import subprocess
import sys
import textwrap
from zipfile import ZipFile


def run(cmd, cwd=None):
    print('\n$', ' '.join(map(str, cmd)))
    subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=True)


def ensure_dependencies():
    run([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        'kaggle',
        'tensorflow>=2.16,<2.21',
        'matplotlib',
        'pandas',
        'scikit-learn',
    ])


def maybe_mount_drive():
    if not MOUNT_DRIVE:
        return
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')


def configure_kaggle():
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    target = kaggle_dir / 'kaggle.json'

    if os.getenv('KAGGLE_USERNAME') and os.getenv('KAGGLE_KEY'):
        payload = {
            'username': os.environ['KAGGLE_USERNAME'],
            'key': os.environ['KAGGLE_KEY'],
        }
        target.write_text(json.dumps(payload))
        target.chmod(0o600)
        print('Configured Kaggle from environment variables.')
        return

    if KAGGLE_JSON.exists():
        shutil.copy2(KAGGLE_JSON, target)
        target.chmod(0o600)
        print('Configured Kaggle from kaggle.json file.')
        return

    raise FileNotFoundError(
        'Kaggle credentials not found. Upload kaggle.json to /content/kaggle.json '
        'or set KAGGLE_USERNAME and KAGGLE_KEY.'
    )


def pull_kernel(kernel_slug, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
    run(['kaggle', 'kernels', 'pull', kernel_slug, '-m', '-p', str(output_dir)])
    assets = sorted(output_dir.iterdir())
    print('Kernel assets:')
    for asset in assets:
        print('-', asset)
    return assets


def infer_dataset_slug_from_kernel_assets(assets):
    metadata_path = next((path for path in assets if path.name == 'kernel-metadata.json'), None)
    if metadata_path is None or not metadata_path.exists():
        return None
    metadata = json.loads(metadata_path.read_text())
    dataset_sources = metadata.get('dataset_sources') or []
    if dataset_sources:
        inferred = dataset_sources[0]
        print('Inferred Kaggle dataset from kernel metadata:', inferred)
        return inferred
    return None


def maybe_download_dataset(dataset_slug, data_dir):
    if not dataset_slug:
        return
    data_dir.mkdir(parents=True, exist_ok=True)
    run(['kaggle', 'datasets', 'download', '-d', dataset_slug, '-p', str(data_dir)])
    zip_files = sorted(data_dir.glob('*.zip'))
    if not zip_files:
        raise FileNotFoundError('No downloaded dataset zip file found.')
    archive = zip_files[-1]
    print('Extracting', archive)
    with ZipFile(archive, 'r') as zf:
        zf.extractall(data_dir)


def maybe_extract_dataset_archive(dataset_archive, data_dir):
    if not dataset_archive:
        return
    archive = Path(dataset_archive)
    if not archive.exists():
        raise FileNotFoundError(f'Dataset archive not found: {archive}')
    data_dir.mkdir(parents=True, exist_ok=True)
    print('Extracting dataset archive', archive)
    with ZipFile(archive, 'r') as zf:
        zf.extractall(data_dir)


def maybe_copy_dataset_dir(dataset_source_dir, data_dir):
    if not dataset_source_dir:
        return
    source_dir = Path(dataset_source_dir)
    if not source_dir.exists():
        raise FileNotFoundError(f'Dataset source directory not found: {source_dir}')
    if source_dir.resolve() == data_dir.resolve():
        print('DATASET_SOURCE_DIR already matches DATA_DIR. No copy needed.')
        return
    data_dir.mkdir(parents=True, exist_ok=True)
    print('Copying dataset contents from', source_dir, 'to', data_dir)
    for item in source_dir.iterdir():
        target = data_dir / item.name
        if target.exists():
            continue
        if item.is_dir():
            shutil.copytree(item, target)
        else:
            shutil.copy2(item, target)


def detect_dataset_layout(data_dir):
    train_dir = data_dir / 'train'
    val_dir = data_dir / 'val'

    if (train_dir / 'real').exists() and (train_dir / 'fake').exists():
        layout = {'train': train_dir}
        if (val_dir / 'real').exists() and (val_dir / 'fake').exists():
            layout['val'] = val_dir
        return layout

    if (data_dir / 'real').exists() and (data_dir / 'fake').exists():
        return {'all': data_dir}

    for path in sorted(data_dir.rglob('*')):
        if not path.is_dir():
            continue
        names = {child.name.lower() for child in path.iterdir() if child.is_dir()}
        if {'real', 'fake'}.issubset(names):
            return {'all': path}

    metadata_candidates = sorted(data_dir.rglob('metadata.csv'))
    for metadata_path in metadata_candidates:
        parent = metadata_path.parent
        for images_dir_name in ('faces_224', 'faces', 'images'):
            images_dir = parent / images_dir_name
            if images_dir.exists() and images_dir.is_dir():
                return {'metadata': metadata_path, 'images_dir': images_dir}

    raise FileNotFoundError(textwrap.dedent(f'''
        Could not detect a dataset layout under {data_dir}.

        Configure one of these at the top of the notebook:
        - KAGGLE_DATASET = 'owner/dataset-slug'
        - DATASET_ARCHIVE = Path('/content/your_dataset.zip')
        - DATASET_SOURCE_DIR = Path('/content/drive/MyDrive/your_dataset_folder')

        Expected folder layout:
        - /content/data/real
        - /content/data/fake
          or
        - /content/data/train/real
        - /content/data/train/fake
        - /content/data/val/real
        - /content/data/val/fake
          or
        - /content/data/metadata.csv
        - /content/data/faces_224/
    ''').strip())


def prepare_dataset(data_dir, dataset_slug):
    maybe_copy_dataset_dir(DATASET_SOURCE_DIR, data_dir)
    maybe_extract_dataset_archive(DATASET_ARCHIVE, data_dir)
    maybe_download_dataset(dataset_slug, data_dir)
    return detect_dataset_layout(data_dir)


ensure_dependencies()
maybe_mount_drive()

WORK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

kernel_assets = []
if PULL_REFERENCE_KERNEL or KAGGLE_DATASET:
    configure_kaggle()
if PULL_REFERENCE_KERNEL:
    kernel_assets = pull_kernel(KAGGLE_KERNEL, WORK_DIR / 'kaggle_kernel')

effective_kaggle_dataset = KAGGLE_DATASET or infer_dataset_slug_from_kernel_assets(kernel_assets)
print('Effective Kaggle dataset =', effective_kaggle_dataset)

dataset_layout = prepare_dataset(DATA_DIR, effective_kaggle_dataset)
print('Detected dataset layout:', dataset_layout)

# =========================
# Training
# =========================
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split

tf.random.set_seed(SEED)
np.random.seed(SEED)


def make_datasets(layout, image_size, batch_size, seed):
    dims = (image_size, image_size)
    if 'metadata' in layout:
        metadata = pd.read_csv(layout['metadata'])
        images_dir = layout['images_dir']

        filename_col = next((column for column in ('videoname', 'filename', 'file', 'image', 'path') if column in metadata.columns), None)
        label_col = next((column for column in ('label', 'class', 'target') if column in metadata.columns), None)

        if filename_col is None or label_col is None:
            raise ValueError(f'Unsupported metadata columns in {layout["metadata"]}: {list(metadata.columns)}')

        def resolve_image_path(name):
            raw_name = str(name)
            stem = Path(raw_name).stem
            candidates = [
                images_dir / raw_name,
                images_dir / f'{stem}.jpg',
                images_dir / f'{stem}.jpeg',
                images_dir / f'{stem}.png',
            ]
            for candidate in candidates:
                if candidate.exists():
                    return str(candidate)
            matches = sorted(images_dir.glob(f'{stem}.*'))
            if matches:
                return str(matches[0])
            return None

        metadata = metadata[[filename_col, label_col]].copy()
        metadata['file_path'] = metadata[filename_col].map(resolve_image_path)
        metadata['label_name'] = metadata[label_col].astype(str).str.strip().str.upper()
        metadata['label_num'] = metadata['label_name'].map({'REAL': 0.0, 'FAKE': 1.0, '0': 0.0, '1': 1.0})
        metadata = metadata.dropna(subset=['file_path', 'label_num']).reset_index(drop=True)
        if metadata.empty:
            raise FileNotFoundError(f'No labeled images could be resolved from {layout["metadata"]} and {images_dir}.')

        stratify = metadata['label_num'] if metadata['label_num'].nunique() > 1 else None
        train_df, val_df = train_test_split(metadata, test_size=0.2, random_state=seed, stratify=stratify)

        def make_tensor_dataset(frame, shuffle):
            paths = frame['file_path'].astype(str).tolist()
            labels = frame['label_num'].astype('float32').tolist()
            dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
            if shuffle:
                dataset = dataset.shuffle(len(paths), seed=seed, reshuffle_each_iteration=True)

            def load_example(path, label):
                image = tf.io.read_file(path)
                image = tf.io.decode_image(image, channels=3, expand_animations=False)
                image = tf.image.resize(image, dims)
                image = tf.cast(image, tf.float32)
                label = tf.reshape(tf.cast(label, tf.float32), (1,))
                return image, label

            dataset = dataset.map(load_example, num_parallel_calls=tf.data.AUTOTUNE)
            dataset = dataset.batch(batch_size)
            dataset = dataset.prefetch(tf.data.AUTOTUNE)
            return dataset

        train_ds = make_tensor_dataset(train_df, shuffle=True)
        val_ds = make_tensor_dataset(val_df, shuffle=False)
        class_names = ['real', 'fake']
        return train_ds, val_ds, class_names

    if 'train' in layout:
        train_ds = tf.keras.utils.image_dataset_from_directory(
            layout['train'],
            labels='inferred',
            label_mode='binary',
            image_size=dims,
            batch_size=batch_size,
            shuffle=True,
            seed=seed,
        )
        if 'val' not in layout:
            raise ValueError('Validation directory missing for explicit train/val layout.')
        val_ds = tf.keras.utils.image_dataset_from_directory(
            layout['val'],
            labels='inferred',
            label_mode='binary',
            image_size=dims,
            batch_size=batch_size,
            shuffle=False,
        )
    else:
        train_ds = tf.keras.utils.image_dataset_from_directory(
            layout['all'],
            labels='inferred',
            label_mode='binary',
            image_size=dims,
            batch_size=batch_size,
            validation_split=0.2,
            subset='training',
            seed=seed,
        )
        val_ds = tf.keras.utils.image_dataset_from_directory(
            layout['all'],
            labels='inferred',
            label_mode='binary',
            image_size=dims,
            batch_size=batch_size,
            validation_split=0.2,
            subset='validation',
            seed=seed,
        )

    class_names = list(train_ds.class_names)
    autotune = tf.data.AUTOTUNE
    train_ds = train_ds.prefetch(autotune)
    val_ds = val_ds.prefetch(autotune)
    return train_ds, val_ds, class_names


def build_model(backbone, image_size, learning_rate):
    backbone = backbone.lower()
    input_shape = (image_size, image_size, 3)

    if backbone == 'custom_cnn':
        inputs = tf.keras.Input(shape=input_shape)
        x = tf.keras.layers.Rescaling(1.0 / 255.0)(inputs)
        x = tf.keras.layers.Conv2D(64, 7, activation='relu', padding='same', kernel_initializer='he_normal')(x)
        x = tf.keras.layers.MaxPooling2D()(x)
        x = tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same', kernel_initializer='he_normal')(x)
        x = tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same', kernel_initializer='he_normal')(x)
        x = tf.keras.layers.MaxPooling2D()(x)
        x = tf.keras.layers.Flatten()(x)
        x = tf.keras.layers.Dense(128, activation='relu', kernel_initializer='he_normal')(x)
        x = tf.keras.layers.Dropout(0.5)(x)
        x = tf.keras.layers.Dense(64, activation='relu', kernel_initializer='he_normal')(x)
        x = tf.keras.layers.Dropout(0.5)(x)
        outputs = tf.keras.layers.Dense(1, activation='sigmoid', name='fake_probability')(x)
        model = tf.keras.Model(inputs, outputs, name='custom_cnn_deepfake')
        base_model = None
    else:
        if backbone == 'xception':
            base_model = tf.keras.applications.Xception(include_top=False, weights='imagenet', input_shape=input_shape)
            preprocess = tf.keras.applications.xception.preprocess_input
        elif backbone == 'vgg16':
            base_model = tf.keras.applications.VGG16(include_top=False, weights='imagenet', input_shape=input_shape)
            preprocess = tf.keras.applications.vgg16.preprocess_input
        elif backbone == 'mobilenetv2':
            base_model = tf.keras.applications.MobileNetV2(include_top=False, weights='imagenet', input_shape=input_shape)
            preprocess = tf.keras.applications.mobilenet_v2.preprocess_input
        elif backbone == 'efficientnetb0':
            base_model = tf.keras.applications.EfficientNetB0(include_top=False, weights='imagenet', input_shape=input_shape)
            preprocess = tf.keras.applications.efficientnet.preprocess_input
        else:
            raise ValueError(f'Unsupported backbone: {backbone}')

        base_model.trainable = False
        inputs = tf.keras.Input(shape=input_shape)
        x = tf.keras.layers.Lambda(preprocess, name='preprocess')(inputs)
        x = base_model(x, training=False)
        x = tf.keras.layers.GlobalAveragePooling2D()(x)
        x = tf.keras.layers.Dropout(0.35)(x)
        x = tf.keras.layers.Dense(256, activation='relu')(x)
        x = tf.keras.layers.Dropout(0.25)(x)
        outputs = tf.keras.layers.Dense(1, activation='sigmoid', name='fake_probability')(x)
        model = tf.keras.Model(inputs, outputs, name=f'{backbone}_deepfake_binary')

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
        ],
    )
    return model, base_model


def merge_histories(*histories):
    merged = {}
    for history in histories:
        if history is None:
            continue
        for key, values in history.history.items():
            merged.setdefault(key, []).extend(list(values))
    return merged


train_ds, val_ds, class_names = make_datasets(
    layout=dataset_layout,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED,
)

(OUTPUT_DIR / 'class_names.json').write_text(json.dumps(class_names, indent=2))
print('Class names:', class_names)
print('Train batches:', tf.data.experimental.cardinality(train_ds).numpy())
print('Val batches:', tf.data.experimental.cardinality(val_ds).numpy())

plt.figure(figsize=(12, 12))
for images, labels in train_ds.take(1):
    for i in range(min(9, len(images))):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        label_idx = int(labels[i].numpy()[0])
        plt.title(class_names[label_idx])
        plt.axis('off')
plt.tight_layout()
plt.show()

model, base_model = build_model(BACKBONE, IMAGE_SIZE, LEARNING_RATE)
model.summary()

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(OUTPUT_DIR / 'best_model.keras'),
        monitor='val_auc',
        mode='max',
        save_best_only=True,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_auc',
        mode='max',
        patience=3,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6,
    ),
]

print('Starting warmup training...')
warmup_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

finetune_history = None
if base_model is not None and FINETUNE_EPOCHS > 0:
    print('Starting fine-tuning...')
    base_model.trainable = True
    if len(base_model.layers) > 20:
        for layer in base_model.layers[:-20]:
            layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE / 10.0),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
        ],
    )
    finetune_history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS + FINETUNE_EPOCHS,
        initial_epoch=len(warmup_history.history.get('loss', [])),
        callbacks=callbacks,
    )

history = merge_histories(warmup_history, finetune_history)
eval_metrics = model.evaluate(val_ds, return_dict=True)
model.save(OUTPUT_DIR / 'final_model.keras')
(OUTPUT_DIR / 'history.json').write_text(json.dumps(history, indent=2))
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(eval_metrics, indent=2))

summary = {
    'kaggle_kernel': KAGGLE_KERNEL,
    'kaggle_dataset': effective_kaggle_dataset,
    'kernel_assets': [str(path) for path in kernel_assets],
    'dataset_layout': {key: str(value) for key, value in dataset_layout.items()},
    'class_names': class_names,
    'backbone': BACKBONE,
    'image_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'finetune_epochs': FINETUNE_EPOCHS,
    'eval_metrics': eval_metrics,
    'best_model': str(OUTPUT_DIR / 'best_model.keras'),
    'final_model': str(OUTPUT_DIR / 'final_model.keras'),
}
(OUTPUT_DIR / 'run_summary.json').write_text(json.dumps(summary, indent=2))

epochs_range = range(1, len(history.get('loss', [])) + 1)

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, history.get('accuracy', []), label='train_accuracy')
plt.plot(epochs_range, history.get('val_accuracy', []), label='val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history.get('loss', []), label='train_loss')
plt.plot(epochs_range, history.get('val_loss', []), label='val_loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss')
plt.legend()
plt.tight_layout()
plt.show()

print('Training complete.')
print(json.dumps(summary, indent=2))
